# Module 03 — Unified Memory

**Released:** PR #4420 + hierarchical `root_scope` isolation (March 2026)

Before the rewrite, CrewAI had three memory stores bolted together — `ShortTermMemory`, `LongTermMemory`, `EntityMemory` — each with its own API, storage, and retrieval rules. The unified `Memory` class replaces all three with **one** store of uniform `MemoryRecord`s and a single query API.

This notebook walks through:

1. What gets stored and how it's scored
2. Writing: `remember` (sync) vs `remember_many` (batched EncodingFlow)
3. Reading: `shallow` vs `deep` recall (and what "deep" actually does)
4. Three isolation patterns: `scope`, `slice`, `root_scope`
5. Plugging memory into **agents**, **crews**, and **flows** — and how the fallback chain works
6. Extending: custom storage, custom embedder, tuning knobs
7. Introspection + cleanup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from crewai.memory import Memory
from showcase.shared import get_llm

memory = Memory(
    llm=get_llm(),
    recency_weight=0.3,
    semantic_weight=0.6,
    importance_weight=0.1,
)
memory

## 1. What's stored — the `MemoryRecord`

Every write produces a `MemoryRecord` with this shape:

| Field | Purpose |
|---|---|
| `id` | UUID |
| `content` | The text |
| `scope` | Path-like namespace, default `/` |
| `categories` | Free-form tags used for filtering |
| `metadata` | Arbitrary dict (ids, timestamps, agent role, …) |
| `importance` | 0–1, feeds the composite score |
| `created_at` / `last_accessed` | Timestamps used for recency decay |
| `embedding` | Vector produced by the embedder |
| `source` | Who wrote it (agent role, tool name, `"user"`, …) |
| `private` | If True, hidden from recall unless `include_private=True` |

You don't construct these manually — `remember()` does. But understanding the fields tells you what `recall()` can filter on.

## 2. The composite score

`recall()` ranks hits with a weighted combination, not raw cosine similarity:

```
composite = w_semantic · semantic + w_recency · decay + w_importance · importance
decay     = 0.5 ^ (age_days / recency_half_life_days)
```

Defaults: `semantic=0.5`, `recency=0.3`, `importance=0.2`, `half_life=30 days`.

- **More semantic, less recency** → archival fact base (docs, policies).
- **More recency, less semantic** → short-term assistant context (current session).
- **Bump importance_weight** → hand-pinned facts survive longer even when old.

These weights are set once at `Memory(...)` construction.

## 3. Writing — `remember` vs `remember_many`

Two paths with different cost profiles.

### `remember(content, …)` — sync, one record
Blocks until the record is persisted. No race with a following `recall()`.

### `remember_many(contents, …)` — batch via `EncodingFlow`
Routes N items through a five-step flow that maximises parallelism:

1. **ONE** embedder call for all items
2. **N** concurrent storage searches (to find near-duplicates in the target scope)
3. **N** concurrent LLM calls — field inference (category/scope/importance when you omit them) + consolidation decisions (merge near-dupes above `consolidation_threshold`, default 0.85)
4. Batch re-embed for any records that got rewritten
5. Bulk write

Use `remember_many` when you have 5+ items to save and can tolerate the LLM inferring missing fields. Use `remember` for a single record or when you need deterministic field values.

In [ ]:
# One-at-a-time — fields are literal
memory.remember(
    "User prefers terse, bullet-point answers with no preamble.",
    categories=["preferences"],
    importance=0.9,
    source="onboarding",
)

# Batched — LLM fills in categories/importance when omitted and
# dedupes against existing records above the consolidation_threshold
memory.remember_many([
    "User works in Python 3.11+ and uses uv, not pip.",
    "User maintains a CrewAI fork and ships patches upstream.",
    "User dislikes over-engineered abstractions in code reviews.",
])

memory.info()  # → ScopeInfo for the root scope

## 4. Reading — `shallow` vs `deep` recall

Same signature, very different cost and behaviour.

### `depth="shallow"` — one embed, one vector query
Good for latency-sensitive paths: chat replies, tool pre-checks, anything in a hot loop. Returns `MemoryMatch` objects scored by the composite formula above.

### `depth="deep"` (default) — routes through `RecallFlow`

The deep path is an LLM-in-the-loop retriever that adapts based on confidence:

```
1.  analyze_query  → LLM decomposes the query into sub-queries
                     + complexity classification (simple | complex)
2.  embed each sub-query
3.  parallel search across candidate scopes (ThreadPool)
4.  compute confidence = max top score
5.  router (decide_depth):
      confidence ≥ confidence_threshold_high (0.8)  → synthesize
      confidence < confidence_threshold_low  (0.5)
          AND exploration_budget > 0                → explore_deeper
      complex query AND confidence < complex_
          query_threshold (0.7) AND budget > 0      → explore_deeper
      else                                          → synthesize
6.  explore_deeper: LLM re-reads top chunks, extracts evidence,
                    records evidence_gaps, decrements budget, loops to 3
7.  synthesize: build MemoryMatch[] with match_reasons + evidence_gaps
```

Every knob in that router is a field on `Memory(...)`: `confidence_threshold_high/low`, `complex_query_threshold`, `exploration_budget`, `query_analysis_threshold` (skip analysis for queries shorter than N chars).

In [ ]:
# Shallow — single embed + single vector search, ~1 LLM-free call
for m in memory.recall("how should I phrase my answer?", limit=3, depth="shallow"):
    print(f"[{m.score:.3f}] {m.record.content}")
    print(f"         reasons={m.match_reasons}")

In [ ]:
# Deep — LLM-in-the-loop retriever
hits = memory.recall(
    "what does the user think about complicated code?",
    limit=3,
    depth="deep",
)
for m in hits:
    print(f"[{m.score:.3f}] {m.record.content}")
    if m.evidence_gaps:
        print(f"         gaps={m.evidence_gaps}")

## 5. Three isolation patterns

Scopes are hierarchical path-strings — `/alice`, `/alice/conversations/2026-04-22`, etc. You pick between three ways to confine reads and writes.

### `memory.scope(path)` — bind a sub-tree
Returns a `MemoryScope`. All reads and writes on it are automatically confined under that path. Cheap — no new storage, just a prefix binding.

### `memory.slice(scopes=[...])` — read-only union
Returns a `MemorySlice` that *only supports recall* (writes raise). Use this for cross-tenant analytics: *"anything either Alice or Bob said about dark mode"*.

### `Memory(root_scope="/tenant-a")` — prefix every op
Applied at construction. Every `remember`/`recall`/`info` silently prepends the root scope. Useful when you want tenant-isolation guaranteed by construction instead of discipline at call sites.

In [ ]:
alice = memory.scope("/alice")
bob = memory.scope("/bob")

alice.remember("Alice prefers dark mode", categories=["ui"], importance=0.7)
bob.remember("Bob prefers light mode", categories=["ui"], importance=0.7)

print("Alice's scope only:")
for m in alice.recall("which theme?", limit=2, depth="shallow"):
    print("  -", m.record.content)

print("Cross-tenant slice (read-only):")
either = memory.slice(scopes=["/alice", "/bob"])
for m in either.recall("which theme?", limit=4, depth="shallow"):
    print(f"  - [{m.record.scope}] {m.record.content}")

## 6. Plugging memory into **agents**, **crews**, and **flows**

`Agent` and `Crew` both expose a `memory=` field with the same type:

```python
memory: bool | Memory | MemoryScope | MemorySlice | None
```

- `True` → construct a default `Memory()` (uses `gpt-4o-mini` + on-disk LanceDB).
- `Memory(...)` instance → bring your own config.
- `MemoryScope` or `MemorySlice` → bound view; isolates that agent/crew to a subtree.
- `None` (default on Agent) → fall back to the **crew's** memory if it exists.

### Fallback order for an agent inside a crew

```
Agent.memory set?     → use it
Agent.memory is None  → use Crew.memory if set
Neither set           → no memory (recall returns [])
```

This lets you do *"one shared memory for the whole crew, except the Critic agent which gets its own scope"*:

```python
critic = Agent(..., memory=crew_memory.scope("/critic"))
other  = Agent(...)   # inherits crew memory

crew = Crew(agents=[critic, other], memory=crew_memory)
```

### Flows

`Flow` doesn't have a built-in `memory=` field — you instantiate `Memory()` inside a Flow step and set it on the instance. Module 03's `MemoryFlow` (below) does this, using `object.__setattr__` because Flow attributes are frozen after construction.

In [ ]:
from crewai import Agent

# Agent-only memory (no crew) — bring a scoped view so this agent can't
# read anyone else's facts.
alice_agent = Agent(
    role="Personalized Assistant",
    goal="Reply in Alice's preferred style",
    backstory="Respects Alice's known preferences.",
    llm=get_llm(),
    memory=memory.scope("/alice"),
)
print("Agent.memory type:", type(alice_agent.memory).__name__)

## 7. Run the Flow

`MemoryFlow` (`src/showcase/flows/memory_flow.py`) wires this together: step 1 seeds per-user preferences into a scoped memory, step 2 runs `Agent.kickoff()` to answer the question using the recalled context. Swap `user_id` to see the scope-isolated reply change.

In [ ]:
from showcase.flows.memory_flow import MemoryFlow

flow = MemoryFlow()
flow.kickoff(inputs={
    "user_id": "user-1",
    "question": "How should I approach adding a new config field?",
})
print(flow.state.reply)

## 8. Extending memory

### Custom storage backend

Subclass `crewai.memory.storage.backend.StorageBackend` and implement the sync + async variants of: `save`, `search`, `delete`, `reset`, `count`, `get_record`, `update`, `list_scopes`, `list_categories`, `list_records`, `get_scope_info`. Then:

```python
from my_pkg import PostgresStorage
memory = Memory(storage=PostgresStorage(dsn="…"), llm=get_llm())
```

Built-in backends: `"lancedb"` (default, embedded), `"qdrant"` (Qdrant edge).

### Custom embedder

Pass any object with `embed(text)` → `list[float]` as `embedder=`. If you omit it, CrewAI picks a default that matches the provider inferred from your LLM.

### Tuning knobs

| Knob | Default | When to change |
|---|---|---|
| `recency_weight` / `semantic_weight` / `importance_weight` | 0.3 / 0.5 / 0.2 | Shift the balance. See §2. |
| `recency_half_life_days` | 30 | Shorter for ephemeral assistants, longer for knowledge bases. |
| `consolidation_threshold` | 0.85 | Cosine similarity above which `remember_many` merges near-dupes into one record. |
| `consolidation_limit` | 5 | Max merges per new write. |
| `confidence_threshold_high` / `_low` | 0.8 / 0.5 | How confident deep recall has to be before it stops exploring. |
| `complex_query_threshold` | 0.7 | Complex queries need higher confidence before they stop exploring. |
| `exploration_budget` | 1 | How many `explore_deeper` loops the router is allowed. |
| `query_analysis_threshold` | 200 | Skip LLM query-analysis for queries shorter than N chars. |
| `read_only` | False | Flip to True to freeze writes (useful for production post-hoc audits). |

### Custom memory extraction

`memory.extract_memories(long_text)` → `list[str]` uses the LLM to pull standalone facts out of a long passage (transcripts, emails). Feed the result into `remember_many`.

## 9. Introspection + cleanup

```python
memory.tree()                    # ASCII view of the scope tree
memory.info("/alice")            # ScopeInfo: record_count, categories, oldest/newest
memory.list_scopes()             # All scopes
memory.list_categories()         # Category → count
memory.list_records(scope="/alice", limit=50)

memory.update(record_id, content="corrected text", importance=1.0)
memory.forget(scope="/alice", categories=["ui"])   # selective delete
memory.reset(scope="/alice")                       # nuke a subtree
memory.drain_writes()            # flush any pending async writes
memory.close()                   # shut the storage backend down cleanly
```

In [ ]:
print(memory.tree())
print("---")
print(memory.list_categories())